# الفصل 38: نماذج توليد الصور: Diffusion

> فهم Diffusion ضروري لأي حديث جاد عن Multimodal الحديث — فهي تقنية التوليد البصري المهيمنة (Stable Diffusion, DALL-E 3, Midjourney, Imagen)، وأصبحت تُدمَج الآن مباشرة مع LLMs في أنظمة توليد الصور بالأوامر النصية داخل...

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 38
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 37: نماذج الرؤية واللغة: VLMs
- **الوسوم:** DDPM، Latent Diffusion، Stable Diffusion، UNet

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch38-diffusion-models.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> فهم Diffusion ضروري لأي حديث جاد عن Multimodal الحديث — فهي تقنية التوليد البصري المهيمنة (Stable Diffusion, DALL-E 3, Midjourney, Imagen)، وأصبحت تُدمَج الآن مباشرة مع LLMs في أنظمة توليد الصور بالأوامر النصية داخل...

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 37: نماذج الرؤية واللغة: VLMs


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الحدس الأساسي
- فهم الصياغة الرياضية
- فهم DDPM


## 1. لماذا يهمّك هذا الموضوع؟

فهم Diffusion ضروري لأي حديث جاد عن Multimodal الحديث — فهي تقنية التوليد البصري المهيمنة (Stable Diffusion, DALL-E 3, Midjourney, Imagen)، وأصبحت تُدمَج الآن مباشرة مع LLMs في أنظمة توليد الصور بالأوامر النصية داخل محادثة واحدة (مثل GPT-4o الأصلي قبل الاستبدال بتوليد موحّد).


## 2. الخريطة الذهنية


```text
Diffusion Model = عملية إتلاف تدريجي (Forward) + تعلّم عكسها (Reverse)

صورة نظيفة x₀ --[إضافة ضوضاء تدريجيًا]--> ... --> ضوضاء خالصة xT
                                                        │
                                              نتعلّم نموذجًا يعكس
                                              العملية خطوة بخطوة
                                                        │
ضوضاء عشوائية xT <--[إزالة ضوضاء تدريجيًا]-- ... <-- صورة مولَّدة x₀
```


## 3. الشرح المتدرج

### 🟢 أ) الحدس الأساسي

تخيّل أنك تأخذ صورة وتضيف إليها ضوضاء غاوسية تدريجيًا عبر آلاف الخطوات حتى تصبح ضوضاء عشوائية بحتة. الفكرة العبقرية لـ **DDPM** (Denoising Diffusion Probabilistic Models): درّب شبكة عصبية (عادة U-Net) على التنبؤ بـ"كمية الضوضاء المُضافة" في كل خطوة، ثم استخدمها لعكس العملية خطوة بخطوة بدءًا من ضوضاء عشوائية خالصة، فتحصل على صورة جديدة متّسقة.

### 🟡 ب) الصياغة الرياضية

**عملية التقدّم (Forward Process)** — سلسلة ماركوف ثابتة (غير مُتعلَّمة):

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t; \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right)
$$

حيث $\beta_t$ جدول تباين صغير يزداد مع $t$. بفضل خاصية الجمع للتوزيعات الغاوسية، يمكننا القفز مباشرة لأي خطوة $t$ بصيغة مغلقة:

$$
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon,\quad \epsilon \sim \mathcal{N}(0, I),\quad \bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s)
$$

**عملية التراجع (Reverse Process)** — هذا ما نتعلّمه:

$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\left(x_{t-1}; \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\right)
$$

**دالة الخسارة المبسّطة (الأهم عمليًا)** — بدلاً من التنبؤ بالمتوسط مباشرة، التنبؤ بالضوضاء نفسها أسهل وأكثر استقرارًا:

$$
\mathcal{L}_{\text{simple}} = \mathbb{E}_{x_0, t, \epsilon}\left[\left\| \epsilon - \epsilon_\theta(x_t, t) \right\|^2\right]
$$

🟢 **طبقة الحدس لهذه الصيغة**: النموذج لا يتعلّم "رسم صورة"، بل يتعلّم مهمة أبسط بكثير: "بالنظر إلى صورة مشوّشة في الخطوة $t$، خمّن ما مقدار الضوضاء العشوائية التي أُضيفت إليها". هذا الإبسيط الهندسي هو سرّ نجاح Diffusion.

### 🟡 ج) الكود الكامل: تطبيق DDPM مبسّط للتدريب والعينة


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiffusionScheduler:
    """جدولة الضوضاء (Noise Schedule) وعمليات forward/reverse الأساسية"""
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def add_noise(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None):
        """الانتقال المباشر لخطوة t (عملية Forward بصيغة مغلقة)"""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_1m_ac = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        x_t = sqrt_ac * x0 + sqrt_1m_ac * noise
        return x_t, noise

    @torch.no_grad()
    def sample_step(self, model, x_t, t, t_index, text_embeds=None):
        """خطوة واحدة من عملية الإزالة التراجعية (Reverse)"""
        beta_t = self.betas[t].view(-1, 1, 1, 1)
        sqrt_1m_ac_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_recip_alpha_t = torch.sqrt(1.0 / self.alphas[t]).view(-1, 1, 1, 1)

        predicted_noise = model(x_t, t, text_embeds)
        model_mean = sqrt_recip_alpha_t * (
            x_t - beta_t * predicted_noise / sqrt_1m_ac_t
        )
        if t_index == 0:
            return model_mean
        else:
            noise = torch.randn_like(x_t)
            posterior_variance = beta_t
            return model_mean + torch.sqrt(posterior_variance) * noise

class SinusoidalTimeEmbedding(nn.Module):
    """ترميز الخطوة الزمنية t — نفس فكرة Positional Encoding من الفصل 11!"""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        freqs = torch.exp(
            -torch.arange(half_dim, device=t.device) * (torch.log(torch.tensor(10000.0)) / half_dim)
        )
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class SimpleConditionalUNet(nn.Module):
    """
    U-Net مبسّط موجَّه نصيًا (Text-Conditioned) — جوهر Stable Diffusion
    يتضمن: Cross-Attention مع نص (نفس آلية الفصل 10!) + ترميز زمني
    """
    def __init__(self, in_channels=4, base_channels=128, text_embed_dim=768, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU()
        )
        self.down1 = self._conv_block(in_channels, base_channels)
        self.down2 = self._conv_block(base_channels, base_channels * 2)

        # نفس MultiHeadAttention من الفصل 10، لكن هنا الاستعلام من الصورة والمفتاح/القيمة من النص
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=base_channels * 2, num_heads=8, batch_first=True,
            kdim=text_embed_dim, vdim=text_embed_dim
        )
        self.up1 = self._conv_block(base_channels * 2, base_channels)
        self.out_conv = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.SiLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.SiLU()
        )

    def forward(self, x, t, text_embeds):
        time_emb = self.time_mlp(t)  # [B, time_dim]
        h = self.down1(x)
        h = self.down2(h)

        B, C, H, W = h.shape
        h_flat = h.view(B, C, H * W).permute(0, 2, 1)  # [B, HW, C] كـ query
        attn_out, _ = self.cross_attn(query=h_flat, key=text_embeds, value=text_embeds)
        h = h_flat.permute(0, 2, 1).view(B, C, H, W) + attn_out.permute(0, 2, 1).view(B, C, H, W)

        h = self.up1(h)
        return self.out_conv(h)

def diffusion_training_step(model, scheduler, x0, text_embeds, optimizer):
    """خطوة تدريب واحدة كاملة"""
    batch_size = x0.shape[0]
    t = torch.randint(0, scheduler.timesteps, (batch_size,), device=x0.device)
    x_t, noise = scheduler.add_noise(x0, t)

    predicted_noise = model(x_t, t, text_embeds)
    loss = F.mse_loss(predicted_noise, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


### 🔴 ج) طبقة التعمّق: Latent Diffusion (سرّ كفاءة Stable Diffusion)

تشغيل Diffusion في فضاء البكسلات الخام (مثلاً 512×512×3) مكلف حسابيًا للغاية. الابتكار المحوري في **Stable Diffusion** هو تشغيل عملية الانتشار بأكملها في **فضاء كامن (Latent Space) مضغوط**:


```text
صورة 512×512×3  --[VAE Encoder]-->  latent 64×64×4  (ضغط 48×!)
                                          │
                              [Diffusion يعمل هنا بالكامل]
                                          │
latent 64×64×4  --[VAE Decoder]-->  صورة 512×512×3 نهائية
```


| المكوّن | الدور | مُدرَّب بشكل مستقل؟ |
|---|---|---|
| **VAE (Encoder/Decoder)** | ضغط/فك ضغط الصورة ↔ latent | نعم، مُدرَّب مسبقًا ومجمّد أثناء تدريب U-Net |
| **Text Encoder (CLIP/T5)** | تحويل النص لـ embeddings للـ cross-attention | عادة مجمّد |
| **U-Net** | عملية الانتشار الفعلية في الـ latent | المُدرَّب الرئيسي |

💡 **Classifier-Free Guidance (CFG)** — التقنية التي تجعل الصور "تتبع النص" بقوة أكبر:

$$
\hat{\epsilon}_\theta = \epsilon_\theta(x_t, \varnothing) + w \cdot \left(\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \varnothing)\right)
$$

حيث $c$ هو شرط النص و $\varnothing$ تمثيل فارغ (unconditional)، و $w$ هو "مقياس التوجيه" (عادة 7-12). كلما زاد $w$، زادت دقّة الاتّباع للنص على حساب التنوّع.


In [ ]:
def classifier_free_guidance_sample(model, x_t, t, text_embeds, null_embeds, guidance_scale=7.5):
    noise_cond = model(x_t, t, text_embeds)
    noise_uncond = model(x_t, t, null_embeds)
    return noise_uncond + guidance_scale * (noise_cond - noise_uncond)


## 4. مثال محلول خطوة بخطوة

نأخذ مثالًا رقميًا صغيرًا لخطوة أمامية واحدة. لنفترض $x_0 = 0.80$، وجدول ضوضاء $\beta_1 = 0.02$، إذن $\alpha_1 = 1 - \beta_1 = 0.98$ و$\bar{\alpha}_1 = 0.98$.

**الخطوة 1 — إضافة الضوضاء:**

$$x_1 = \sqrt{\bar{\alpha}_1}\,x_0 + \sqrt{1-\bar{\alpha}_1}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0,1)$$

نختار عشوائيًا $\epsilon = 0.50$:

$$x_1 = \sqrt{0.98}\times 0.80 + \sqrt{0.02}\times 0.50 \approx 0.9899\times0.80 + 0.1414\times0.50 \approx 0.8626$$

**الخطوة 2 — ماذا تتعلّم الشبكة؟** تُعطى لها $x_1$ والخطوة $t=1$، وتُطلب منها التنبؤ بـ$\epsilon$؛ الهدف هو $0.50$. الخسارة هي $(\hat{\epsilon}-0.50)^2$.

**الخطوة 3 — العينة العكسية:** بعد تدريب الشبكة، نبدأ من ضوضاء خالصة $x_T\sim\mathcal{N}(0,1)$ ونطبّق الخطوة العكسية repeatedly حتى نصل إلى $x_0$. في DDIM نستخدم نفس المعادلة دون إضافة ضوضاء جديدة، ولهذا يمكن تقليل الخطوات من 1000 إلى 50.

**ملاحظة:** في Stable Diffusion لا تُطبَّق هذه العملية على البكسل مباشرة، بل على تمثيل كامن من VAE؛ ثم يُفكّ التمثيل إلى صورة في النهاية.


## 5. نصائح وحيل

> 💡 **ابدأ بـ DDIM أو DPM-Solver++ في الإنتاج**: 20-30 خطوة تكفي غالبًا، وكل خطوة أقل تعني زمن استجابة أقل وتكلفة GPU أقل.

> 💡 **اضبط CFG Scale بعناية**: القيم 7-9 شائعة، لكن الزيادة الكبيرة تُنتج صورًا مشبعة/متكلفة وتُقلّل التنوع. اختبر 5، 7، 9 على نفس البذرة.

> 💡 **ثبّت البذرة (Seed) أثناء المقارنة**: لا يمكن الحكم على أثر تغيير prompt أو sampler إذا تغيّرت البذرة في كل تجربة.

> 💡 **استخدم Negative Prompt بوعي**: هو ليس سحرًا؛ فعّاليته تعتمد على CFG وعلى صياغة النموذج. ابدأ بقائمة قصيرة (blurry, low quality) بدلًا من قائمة طويلة متناقضة.

> 💡 **قِس الجودة بمعايير متعددة**: CLIPScore أو FID لا يكفيان وحدهما؛ أضف مراجعة بشرية لعينة صغيرة، خاصة للصور العربية/النصية.


## 6. أخطاء شائعة ومصائد

> ⚠️ **استخدام 1000 خطوة في الإنتاج**: DDPM الأصلي بطيء جدًا. الانتقال إلى DDIM/DPM-Solver++ هو أول تحسين يجب أن تفعله.

> ⚠️ **خلط جدول الضوضاء بين التدريب والاستدلال**: إذا دُرِّب النموذج على $\beta$ خطي واستُخدم scheduler آخر في العينة، تنهار الجودة. استخدم نفس الإعداد ونفس تعريف $\bar{\alpha}$.

> ⚠️ **تجاهل VAE في Latent Diffusion**: لا تفكّ التمثيل الكامن مباشرة إلى صورة؛ يجب تمريره عبر VAE decoder. كما أن VAE نفسه قد يُدخل تشوهات في النصوص الصغيرة والوجوه.

> ⚠️ **CFG مرتفع جدًا لتعويض prompt ضعيف**: هذا يُنتج صورًا "محروقة" ويُقلّل التنوع. حسّن الوصف أولًا، ثم اضبط CFG.

> ⚠️ **الحكم على النموذج من 3-4 صور**: التوليد عشوائي؛ استخدم 20-50 صورة على الأقل مع بذور متعددة قبل الحكم.


## 7. جدول مقارنة

| الأسلوب | أمثلة | السرعة | الجودة | التحكّم |
|---|---|---|---|---|
| **GANs** | StyleGAN | سريع جدًا (خطوة واحدة) | عالية لكن أقل تنوعًا | صعب التحكّم النصي |
| **Diffusion (Pixel Space)** | Imagen, DALL-E 2 | بطيء (خطوات كثيرة) | الأعلى | ممتاز عبر النص |
| **Latent Diffusion** | Stable Diffusion | متوسط-سريع | عالية جدًا | ممتاز وأرخص |
| **Autoregressive** | DALL-E 1, Parti | بطيء جدًا | جيدة | ممتاز (نفس آلية LLM) |
| **Consistency Models** | LCM | سريع جدًا (1-4 خطوات) | جيدة جدًا | جيد |


## 8. ملخص أفضل الممارسات

- استخدم **DDIM** أو **DPM-Solver++** بدلاً من Ancestral Sampling العادي لتقليل خطوات العينة من 1000 إلى 20-50 دون فقدان جودة ملموس.
- لمهام التحكّم الدقيق (حواف، وضعيات، عمق) استخدم **ControlNet** — فرع إضافي مُدرَّب يُضاف لأوزان U-Net المجمّدة.
- راقب دائمًا موازنة `guidance_scale`: قيم عالية جدًا (>15) تُنتج صورًا "محروقة" الألوان (over-saturated).

---


## 9. أسئلة مقابلات

**س1: ما الفرق الجوهري بين التنبؤ بالضوضاء $\epsilon$ والتنبؤ بالصورة النظيفة $x_0$؟**

**الإجابة النموذجية:** كلاهما صياغة مكافئة رياضيًا تقريبًا، لكن التنبؤ بالضوضاء هو الأكثر شيوعًا لأن الضوضاء هي الهدف البسيط الذي نعرفه في كل خطوة، ولأن خسارة MSE عليه تعمل جيدًا عبر جميع مستويات $t$. التنبؤ بـ$x_0$ قد يكون أفضل في الخطوات المتأخرة، وهناك صيغ تجمع بينهما (v-prediction) لتحسين الاستقرار.

**س2: لماذا يعمل Latent Diffusion بكفاءة أعلى من Diffusion في البكسل؟**

**الإجابة النموذجية:** VAE يضغط الصورة إلى فضاء كامن أصغر بكثير (مثلًا 64×64×4 بدلًا من 512×512×3)، فتصبح كل خطوة U-Net أرخص حسابيًا وذاكرة. معظم المعلومات الإدراكية المهمة محفوظة، بينما التفاصيل عالية التردد غير الضرورية تُختصر. هذا يسمح بتدريب وعيّنة على GPU واحدة بدلًا من عنقود كامل.

**س3: ما وظيفة Classifier-Free Guidance، ولماذا قد تكون ضارّة؟**

**الإجابة النموذجية:** CFG يمزج بين تنبؤ شرطي (مع النص) وغير شرطي (بدون نص): $\hat{\epsilon} = \epsilon_\varnothing + w(\epsilon_c - \epsilon_\varnothing)$. زيادة $w$ تُقوّي الالتزام بالنص لكنها تُقلّل التنوع وقد تُنتج ألوانًا مشبعة وتشوهات. لذلك تُختار $w$ تجريبيًا، وغالبًا بين 5 و9.


## 10. تمرين عملي

1. نفّذ `ddim_sample_step` بجانب `ddpm_sample_step`، وقارن 50 خطوة DDIM مقابل 1000 خطوة DDPM على نفس البذرة.
2. احسب حجم التمثيل الكامن لصورة 512×512 بعد VAE بمعامل ضغط 8× في كل بُعد، وقارنه بحجم الصورة الأصلية.
3. جرّب CFG = 1، 3، 7، 12 على نفس prompt والبذرة، وقيّم الالتزام بالنص والتنوع.
4. أضف Negative Prompt ثابتًا وقِس أثره على 20 صورة.
5. اكتب تقريرًا من صفحة واحدة: أي sampler وأي عدد خطوات وأي CFG تختار للإنتاج ولماذا.


## 11. مراجع للتعمّق

- Ho, J., Jain, A., & Abbeel, P. (2020). *Denoising Diffusion Probabilistic Models*. NeurIPS.
- Song, J., Meng, C., & Ermon, S. (2021). *Denoising Diffusion Implicit Models*. ICLR.
- Rombach, R., et al. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. CVPR.
- Ramesh, A., et al. (2022). *Hierarchical Text-Conditional Image Generation with CLIP Latents* (DALL-E 2).
- Podell, D., et al. (2023). *SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis*.
- Ho, J., & Salimans, T. (2022). *Classifier-Free Diffusion Guidance*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 38: نماذج توليد الصور: Diffusion.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
